# 01: Data Exploration & Quality Analysis

This notebook performs initial exploration of the Olist Brazilian E-Commerce dataset to understand the schema, validate data quality, identify join keys, and document decisions for downstream feature engineering and modelling.

**Outputs:**
- Data quality reports (HTML) saved to `data_reports/`
- Key findings that inform the feature engineering pipeline in `02_feature_engineering.ipynb`

In [3]:
import pandas as pd
import numpy as np
import os

In [54]:
data_dir = '../datasets'
files = os.listdir(data_dir)
print(f"Total files: {len(files)}\n")
for f in sorted(files):
    size_mb = os.path.getsize(os.path.join(data_dir, f)) / (1024 * 1024)
    print(f"{f:50s} {size_mb:8.2f} MB")

Total files: 11

olist_closed_deals_dataset.csv                         0.16 MB
olist_customers_dataset.csv                            8.62 MB
olist_geolocation_dataset.csv                         58.44 MB
olist_marketing_qualified_leads_dataset.csv            0.67 MB
olist_order_items_dataset.csv                         14.72 MB
olist_order_payments_dataset.csv                       5.51 MB
olist_order_reviews_dataset.csv                       13.78 MB
olist_orders_dataset.csv                              16.84 MB
olist_products_dataset.csv                             2.27 MB
olist_sellers_dataset.csv                              0.17 MB
product_category_name_translation.csv                  0.00 MB


##  Table Classification

Based on the assessment instructions and initial inspection, I classify the 11 files into three tiers:

**Core Tables (used for primary modelling pipeline):**
| Table | Role | Description |
|---|---|---|
| `olist_customers_dataset` | Lead | All potential customers with location data |
| `olist_orders_dataset` | Purchase | Order-level data with timestamps and status |
| `olist_order_items_dataset` | Browsing/Cart | Item-level details per order (price, freight) |

**Supporting Tables (enrich core tables with additional signal):**
| Table | Role | Description |
|---|---|---|
| `olist_order_payments_dataset` | Payment behavior | Payment method, installments, value per order |
| `olist_order_reviews_dataset` | Engagement | Review scores and text per order |
| `olist_products_dataset` | Product metadata | Category, dimensions, weight |
| `product_category_name_translation` | Lookup | Portuguese → English category names |

**Excluded Tables (not relevant to customer purchase prediction):**
| Table | Reason for Exclusion |
|---|---|
| `olist_geolocation_dataset` | 58MB, customer city/state already available in customers table |
| `olist_sellers_dataset` | Seller-side data, low predictive value for customer behavior |
| `olist_closed_deals_dataset` | B2B seller acquisition funnel, uses `mql_id`/`seller_id` not `customer_id` |
| `olist_marketing_qualified_leads_dataset` | Same as above — seller marketing funnel, not customer behavior |

In [9]:
def get_file_details(filename: str):
    filename = data_dir + "/" + filename
    print(f"Processing {filename}")
    df = pd.read_csv(filename)
    print(f"Shape: {df.shape}")
    print(f"Columns:\n {df.columns}")
    print(f"dtypes:\n {df.dtypes}")
    print(f"Head:\n {df.head()}")
    print("="*100)

## 3. Core Tables : Schema Exploration

In [10]:
main_files = ['olist_customers_dataset.csv', 'olist_order_items_dataset.csv', 'olist_orders_dataset.csv']

In [11]:
for file in main_files:
    get_file_details(file)

Processing ./../datasets/olist_customers_dataset.csv
Shape: (99441, 5)
Columns:
 Index(['customer_id', 'customer_unique_id', 'customer_zip_code_prefix',
       'customer_city', 'customer_state'],
      dtype='object')
dtypes:
 customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object
Head:
                         customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   
3  b2b6027bc5c5109e529d4dc6358b12c3  259dac757896d24d7702b9acbbff3f3c   
4  4f2d8ab171c80ec8364f7c12e35b23ad  345ecd01c38d18a9036ed96c73b8d066   

   customer_zip_code_prefix          customer_city customer_state  
0                     14409                 franca             SP  
1 

In [24]:
customers_df = pd.read_csv(data_dir + "/" + "olist_customers_dataset.csv")
order_items_df = pd.read_csv(data_dir + "/" + "olist_order_items_dataset.csv")
orders_df = pd.read_csv(data_dir + "/" + "olist_orders_dataset.csv")

In [59]:
core_tables = {
    'Customers': customers_df,
    'Orders': orders_df,
    'Order Items': order_items_df
}

for name, df in core_tables.items():
    print(f"{'='*60}")
    print(f" {name} — Shape: {df.shape}")
    print(f"{'='*60}")
    print(f"\nColumns & Types:")
    print(df.dtypes)
    print(f"\nFirst 3 rows:")
    print(df.head(3))
    print()
    

 Customers — Shape: (99441, 5)

Columns & Types:
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object

First 3 rows:
                        customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   

   customer_zip_code_prefix          customer_city customer_state  
0                     14409                 franca             SP  
1                      9790  sao bernardo do campo             SP  
2                      1151              sao paulo             SP  

 Orders — Shape: (99441, 8)

Columns & Types:
order_id                         object
customer_id                      object
order_status                     object
order_purchase_

### Core Table Observations

**Customers (99,441 rows × 5 cols):**
- Two ID columns: `customer_id` (per-order identifier) and `customer_unique_id` (true person)
- Location data: zip code, city, state
- One row per customer-order occurrence

**Orders (99,441 rows × 8 cols):**
- Same row count as customers — suggests 1:1 mapping with `customer_id`
- Contains full order lifecycle timestamps: purchase → approved → shipped → delivered
- `order_status` captures lifecycle state (delivered, canceled, etc.)
- All timestamp columns are strings — need datetime conversion

**Order Items (112,650 rows × 7 cols):**
- More rows than orders → one-to-many (multiple items per order)
- Contains `price` and `freight_value` per item
- Links to products via `product_id` and sellers via `seller_id`

## 4. Supporting Tables : Schema Exploration

In [56]:
# Load supporting tables
order_payments_df = pd.read_csv(f'{data_dir}/olist_order_payments_dataset.csv')
order_reviews_df = pd.read_csv(f'{data_dir}/olist_order_reviews_dataset.csv')
products_df = pd.read_csv(f'{data_dir}/olist_products_dataset.csv')
product_translation_df = pd.read_csv(f'{data_dir}/product_category_name_translation.csv')


In [57]:
support_tables = {
    'Order Payments': order_payments_df,
    'Order Reviews': order_reviews_df,
    'Products': products_df,
    'Category Translation': product_translation_df
}

In [58]:
for name, df in support_tables.items():
    print(f"{'='*60}")
    print(f" {name} — Shape: {df.shape}")
    print(f"{'='*60}")
    print(f"\nColumns & Types:")
    print(df.dtypes)
    print(f"\nFirst 3 rows:")
    print(df.head(3))
    print()

 Order Payments — Shape: (103886, 5)

Columns & Types:
order_id                 object
payment_sequential        int64
payment_type             object
payment_installments      int64
payment_value           float64
dtype: object

First 3 rows:
                           order_id  payment_sequential payment_type  \
0  b81ef226f3fe1789b1e8b2acac839d17                   1  credit_card   
1  a9810da82917af2d9aefd1278f1dcfa0                   1  credit_card   
2  25e8ea4e93396b6fa0d3dd708e76c1bd                   1  credit_card   

   payment_installments  payment_value  
0                     8          99.33  
1                     1          24.39  
2                     1          65.71  

 Order Reviews — Shape: (99224, 7)

Columns & Types:
review_id                  object
order_id                   object
review_score                int64
review_comment_title       object
review_comment_message     object
review_creation_date       object
review_answer_timestamp    object
dtype: obje

### Supporting Table Observations

**Order Payments (103,886 rows × 5 cols):**
- More rows than orders → one-to-many (some orders have multiple payment methods)
- `payment_type`: credit_card, boleto, voucher, debit_card
- `payment_installments`: financing behavior signal
- `payment_sequential`: tracks multiple payments per order

**Order Reviews (99,224 rows × 7 cols):**
- Nearly 1:1 with orders (slightly fewer — not all orders reviewed)
- `review_score`: 1-5 star rating
- `review_comment_title` and `review_comment_message`: free text (often NaN)

**Products (32,951 rows × 9 cols):**
- One row per product
- `product_category_name` in Portuguese
- Physical attributes: weight, dimensions, photos count

**Category Translation (71 rows × 2 cols):**
- Simple lookup: Portuguese → English category names

## Join Key Validation & Cardinality Checks

Before merging tables, we must verify the grain of each table and understand the join relationships. Getting this wrong leads to row duplication and corrupted aggregations.

In [61]:
# Customers: is customer_id unique? Is customer_unique_id unique?
print(f"\n--- Customers ---")
print(f"Total rows:                {len(customers_df)}")
print(f"Unique customer_id:        {customers_df['customer_id'].nunique()}")
print(f"Unique customer_unique_id: {customers_df['customer_unique_id'].nunique()}")

# Orders: is order_id unique?
print(f"\n--- Orders ---")
print(f"Total rows:         {len(orders_df)}")
print(f"Unique order_id:    {orders_df['order_id'].nunique()}")
print(f"Unique customer_id: {orders_df['customer_id'].nunique()}")

# Order Items: how many per order?
print(f"\n--- Order Items ---")
print(f"Total rows:       {len(order_items_df)}")
print(f"Unique order_id:  {order_items_df['order_id'].nunique()}")
print(f"Avg items/order:  {len(order_items_df) / order_items_df['order_id'].nunique():.2f}")

# Payments: how many per order?
print(f"\n--- Payments ---")
print(f"Total rows:       {len(order_payments_df)}")
print(f"Unique order_id:  {order_payments_df['order_id'].nunique()}")
print(f"Avg payments/order: {len(order_payments_df) / order_payments_df['order_id'].nunique():.2f}")

# Reviews: how many per order?
print(f"\n--- Reviews ---")
print(f"Total rows:       {len(order_reviews_df)}")
print(f"Unique order_id:  {order_reviews_df['order_id'].nunique()}")


--- Customers ---
Total rows:                99441
Unique customer_id:        99441
Unique customer_unique_id: 96096

--- Orders ---
Total rows:         99441
Unique order_id:    99441
Unique customer_id: 99441

--- Order Items ---
Total rows:       112650
Unique order_id:  98666
Avg items/order:  1.14

--- Payments ---
Total rows:       103886
Unique order_id:  99440
Avg payments/order: 1.04

--- Reviews ---
Total rows:       99224
Unique order_id:  98673


### Cardinality Findings

| Table | Key | Unique Values | Rows | Relationship |
|---|---|---|---|---|
| Customers | `customer_id` | 99,441 | 99,441 | 1:1 per row |
| Customers | `customer_unique_id` | 96,096 | 99,441 | ~3% repeat buyers |
| Orders | `order_id` | 99,441 | 99,441 | 1:1 per row |
| Order Items | `order_id` | 98,666 | 112,650 | 1:many (~1.14 items/order) |
| Payments | `order_id` | 99,440 | 103,886 | 1:many (~1.04 payments/order) |
| Reviews | `order_id` | 98,673 | 99,224 | ~1:1 (few multi-review orders) |

**Critical insight:** `customer_unique_id` is the true user identifier. 96,096 unique people placed 99,441 orders, meaning ~3% are repeat purchasers. This repeat purchase behavior is exactly what our model needs to predict.

**Merge strategy implication:** One-to-many tables (items, payments) must be aggregated to order level BEFORE merging with orders, to prevent row explosion.

In [63]:
# Order status distribution
print("Order Status Distribution:")
print(orders_df['order_status'].value_counts())

Order Status Distribution:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [65]:
print(f"\nDelivered: {(orders_df['order_status'] == 'delivered').mean()*100:.1f}%")


Delivered: 97.0%


In [67]:
# Date range of the data
orders_df['order_purchase_timestamp'] = pd.to_datetime(orders_df['order_purchase_timestamp'])
print(f"Date range: {orders_df['order_purchase_timestamp'].min()} to {orders_df['order_purchase_timestamp'].max()}")

Date range: 2016-09-04 21:15:19 to 2018-10-17 17:30:18


In [68]:
# Monthly order volume
orders_df['order_month'] = orders_df['order_purchase_timestamp'].dt.to_period('M')
print(f"\nMonthly order volumes (last 6 months):")
print(orders_df['order_month'].value_counts().sort_index().tail(6))


Monthly order volumes (last 6 months):
order_month
2018-05    6873
2018-06    6167
2018-07    6292
2018-08    6512
2018-09      16
2018-10       4
Freq: M, Name: count, dtype: int64


### Temporal Observations

- Data spans **September 2016 to October 2018** (~25 months)
- Order volume is relatively stable at ~6,000-7,500 orders/month through August 2018
- Sharp dropoff in September 2018 (16 orders) and October 2018 (4 orders), dataset is truncated
- This truncation directly impacts our target window selection (see `02_feature_engineering.ipynb`)

## Data Quality Reports

Automated profiling reports generated using `ydata-profiling` for each key source file. Reports are saved as HTML files in `data_reports/` and provide:
- Column-level null counts and percentages
- Value distributions and frequency analysis
- Correlation detection
- Duplicate detection

In [70]:
from ydata_profiling import ProfileReport

reports_config = {
    'olist_customers_dataset.csv': 'Customers Dataset',
    'olist_orders_dataset.csv': 'Orders Dataset',
    'olist_order_items_dataset.csv': 'Order Items Dataset',
    'olist_order_payments_dataset.csv': 'Order Payments Dataset',
    'olist_order_reviews_dataset.csv': 'Order Reviews Dataset',
    'olist_products_dataset.csv': 'Products Dataset',
    'product_category_name_translation.csv': 'Product Category Translation',
}

for filename, title in reports_config.items():
    print(f"Generating report for: {title}...")
    df = pd.read_csv(f'{data_dir}/{filename}')
    profile = ProfileReport(df, title=title, minimal=True)
    report_name = filename.replace('.csv', '_report.html')
    profile.to_file(f'../data_reports/{report_name}')
    print(f"  Saved: {report_name}")

print("\nAll reports generated!")

Generating report for: Customers Dataset...


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:00<00:00, 11.75it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

...


100%|█████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00, 33288.13it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

  Saved: product_category_name_translation_report.html

All reports generated!


## Key Findings & Decisions

### Data Quality
- No primary keys contain null values
- No dataset-level duplicates detected
- Expected null patterns: not all orders have reviews, timestamps vary by order status
- ~775 orders have no corresponding items in the items table (data gap in source)

### Entity Relationships
- `orders` is the central hub table — all other tables join through `order_id`
- `customers` joins to `orders` via `customer_id` (1:1)
- `order_items`, `payments` are one-to-many — require aggregation before merging

### Key Decisions for Downstream Pipeline
1. **Unit of prediction:** `customer_unique_id` (the real person, not `customer_id`)
2. **Target window:** 90 days (July 1 — October 1, 2018) — chosen because data is truncated after August 2018, and a 30-day window yielded insufficient positive examples
3. **Feature window:** All data before July 1, 2018 (~22 months of history)
4. **Order status handling:** Canceled and unavailable orders excluded from positive conversion labels
5. **Excluded tables:** Geolocation (redundant), sellers (low signal), closed deals & marketing leads (seller-side B2B funnel)
6. **Aggregation strategy:** Aggregate one-to-many tables to order level first, then merge, then aggregate to customer level — prevents row explosion and corrupted calculations